In [7]:
#A function that returns a pandas DataFrame indexed by Region or Country and Year, with columns giving counts of people in different age-sex group
#3. [A] Population DataFrames

In [38]:
import pandas as pd
import re

In [39]:
## If import fails with "ModuleNotFoundError"
## uncomment below & try again
%pip install wbdata

import wbdata

Note: you may need to restart the kernel to use updated packages.


In [89]:
SOURCE = 40 # "Population estimates and projections
indicators = wbdata.get_indicators(source=SOURCE)
indicators

id                 name
-----------------  -------------------------------------------------------------------
SH.DTH.0509        Number of deaths ages 5-9 years
SH.DTH.0514        Number of deaths ages 5-14 years
SH.DTH.1014        Number of deaths ages 10-14 years
SH.DTH.1019        Number of deaths ages 10-19 years
SH.DTH.1519        Number of deaths ages 15-19 years
SH.DTH.2024        Number of deaths ages 20-24 years
SH.DTH.IMRT        Number of infant deaths
SH.DTH.IMRT.FE     Number of infant deaths, female
SH.DTH.IMRT.MA     Number of infant deaths, male
SH.DTH.MORT        Number of under-five deaths
SH.DTH.MORT.FE     Number of under-five deaths, female
SH.DTH.MORT.MA     Number of under-five deaths, male
SH.DTH.NMRT        Number of neonatal deaths
SH.DYN.0509        Probability of dying among children ages 5-9 years (per 1,000)
SH.DYN.0514        Probability of dying at age 5-14 years (per 1,000 children age 5)
SH.DYN.1014        Probability of dying among adolescents ages 1

In [105]:
SOURCE = 40  # Population estimates and projections

indicators_df = pd.DataFrame(wbdata.get_indicators(source=SOURCE))
indicators = indicators_df['id']
indicators.head()

0    SH.DTH.0509
1    SH.DTH.0514
2    SH.DTH.1014
3    SH.DTH.1019
4    SH.DTH.1519
Name: id, dtype: object

In [103]:
SOURCE = 40 # "Population estimates and projections
indicators = pd.DataFrame(wbdata.get_indicators(source=SOURCE))

all_indicators = (
    indicators
    .str.startswith('SP.POP')
    .unique()
    .tolist()
)


all_indicators

[False, True]

In [96]:
def build_age_sex_population_df(source):
    """
    Returns a DataFrame indexed by Country/Region and Year,
    with columns for ALL age/sex population counts.
    Column names are formatted like: 'F 5-9', 'M 12', 'F 65+'.
    """

    # Step 1: pull ALL population indicators
    indicators = ["SP.POP"]

    df = get_dataframe(source, indicators)

    # Standardize column names (adjust if your data uses 'region' instead)
    df = df.rename(columns={
        'country': 'Country',
        'year': 'Year'
    })

    # Step 2: keep only age/sex population COUNT indicators
    df = df[df['indicator'].str.startswith("SP.POP")]
    df = df[~df['indicator'].str.contains("ZS|5Y|IN.ZS")]

    # Step 3: create clean age/sex labels
    def parse_indicator(code):
        sex = "F" if ".FE" in code else "M"

        # Single-year ages (AG12)
        ag_match = re.search(r"AG(\d{2})", code)
        if ag_match:
            age = ag_match.group(1).lstrip("0")
            return f"{sex} {age}"

        # Age ranges (0509, 2024, etc.)
        range_match = re.search(r"\.(\d{2})(\d{2})\.", code)
        if range_match:
            a1, a2 = range_match.groups()
            return f"{sex} {int(a1)}-{int(a2)}"

        # Open-ended groups (65UP, 80UP)
        up_match = re.search(r"\.(\d+)UP\.", code)
        if up_match:
            return f"{sex} {up_match.group(1)}+"

        return None

    df['age_sex'] = df['indicator'].apply(parse_indicator)
    df = df[df['age_sex'].notna()]

    # Step 4: reshape to wide format
    df = (
        df.pivot_table(
            index=['Country', 'Year'],
            columns='age_sex',
            values='value'
        )
        .sort_index()
    )

    return df
    
df = build_age_sex_population_df(source=wbdata)
df.head()

NameError: name 'get_dataframe' is not defined

In [37]:
def population_by_age_sex(source, indicators):
    """
    Returns a DataFrame indexed by Country (or Region) and Year,
    with columns for age-sex population counts.
    """

    # Get raw data using provided function
    df = get_dataframe(source, indicators)

    # Standardize column names (adjust if needed)
    df = df.rename(columns={
        'country': 'Country',
        'year': 'Year'
    })

    # Set index as required by prompt
    df = df.set_index(['Country', 'Year']).sort_index()

    return df

In [28]:
def rename_age_sex_indicators(df):
    new_names = {}

    for col in df.columns:
        if col.startswith("SP.POP"):
            parts = col.split(".")

            # Sex
            sex = "Female" if "FE" in parts else "Male" if "MA" in parts else ""

            # Age group
            if "AG" in col:  # single year of age
                age = col.split("AG")[1][:2].lstrip("0")
                age_label = f"Age {age}"
            elif "UP" in col:
                age_label = col.split(".")[2].replace("UP", "+")
            else:
                age_label = col.split(".")[2].replace("00", "0").replace("04", "4")

            new_names[col] = f"{sex} {age_label}".strip()

    return df.rename(columns=new_names)


In [97]:

wbdata.get_dataframe(variable_labels,parse_dates=True)

World Population
country                     date                        
Africa Eastern and Southern 2024-01-01       769280888.0
                            2023-01-01       750491370.0
                            2022-01-01       731821393.0
                            2021-01-01       713090928.0
                            2020-01-01       694446100.0
...                                                  ...
Zimbabwe                    1964-01-01         4320006.0
                            1963-01-01         4185877.0
                            1962-01-01         4055959.0
                            1961-01-01         3930401.0
                            1960-01-01         3809389.0

[17290 rows x 1 columns]